# 04 — Feature Engineering and Tree Model Improvement

## Objective

This notebook continues from the baseline modeling results.

The goal is to improve tree-based model performance by adding row-wise engineered features and validating whether these features improve cross-validation results.

This notebook covers:

- 2.1 Feature Engineering Pipeline
- 2.2 Feature Validation Loop
- Random Forest improvement
- hyperparameter tuning as an extension

---

# Improvement Strategy

We will improve tree-based models using three main ideas:

## 1. Rebuild the Random Forest Baseline

We first recreate the Random Forest model from the previous notebook.

This gives us a reference point for comparison.

---

## 2. Add Feature Engineering

Because the dataset has anonymous sparse features, we create row-level statistical features such as:

- number of non-zero values
- row sum
- row mean
- row standard deviation
- row maximum value

These features may summarize useful structural information.

---

## 3. Tune Model Hyperparameters

We then adjust Random Forest settings such as:

- number of trees
- tree depth
- minimum samples per leaf
- number of features considered at each split

The goal is to improve performance while avoiding overfitting.

---

# Expected Outcome

By the end of this notebook, we should understand:

- whether engineered row-level features improve performance
- whether Random Forest tuning improves performance
- which settings produce the most stable validation results
- whether tree-based models remain the best direction for this dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_validate, GridSearchCV, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

# 1. Load Data and Recreate Baseline Setup

We begin by loading the same dataset and applying the same basic preprocessing used in the previous notebook.

This ensures that results in this notebook are comparable to the previous Random Forest result.

The steps are:

- load train and test data
- separate features and target
- remove ID columns
- remove constant features
- recreate the same KFold validation setup

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train["target"]
X = train.drop(columns=["ID", "target"])
X_test = test.drop(columns=["ID"])

feature_variances = X.var()
constant_cols = feature_variances[feature_variances == 0].index

X = X.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Removed constant features:", len(constant_cols))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

Removed constant features: 256
X shape: (4459, 4735)
X_test shape: (49342, 4735)


In [4]:
def rmsle(y_true, y_pred):

    y_pred = np.maximum(0, y_pred)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

# 2. Rebuild Random Forest Baseline

Before improving the model, we first recreate the Random Forest baseline from the previous notebook.

This gives us a reference point.

Any feature engineering or tuning should be compared against this baseline.

In [6]:
baseline_forest = RandomForestRegressor(
    n_estimators=50,
    max_depth=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

baseline_scores = cross_validate(
    baseline_forest,
    X,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    ),
    return_train_score=True
)

baseline_predictions = cross_val_predict(
    baseline_forest,
    X,
    y,
    cv=kf
)


baseline_mae = -baseline_scores["test_neg_mean_absolute_error"]
baseline_rmse = -baseline_scores["test_neg_root_mean_squared_error"]
baseline_r2 = baseline_scores["test_r2"]
baseline_rmsle = rmsle(
    y,
    baseline_predictions
)

print("Baseline Random Forest")
print("MAE:", baseline_mae.mean())
print("RMSE:", baseline_rmse.mean())
print("R²:", baseline_r2.mean())
print("RMSLE:", baseline_rmsle)

Baseline Random Forest
MAE: 5031415.062305139
RMSE: 7262026.002143969
R²: 0.21644454014504794
RMSLE: 1.9116026187748898


## 3 Feature Engineering Pipeline

We now create row-wise statistical features.

Because the original feature names are anonymous, it is difficult to interpret individual columns directly.

Instead, we summarize each row using simple statistical features.

These features describe the overall structure of each observation.

Created features:

- row_mean
- row_sum
- row_std
- row_min
- row_max
- non_zero_count

The goal is to test whether row-level structure improves model performance.

In [7]:
class RowFeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X_engineered = X.copy()

        X_engineered["row_nonzero_count"] = (X != 0).sum(axis=1)
        X_engineered["row_sum"] = X.sum(axis=1)
        X_engineered["row_mean"] = X.mean(axis=1)
        X_engineered["row_std"] = X.std(axis=1)
        X_engineered["row_max"] = X.max(axis=1)
        X_engineered["row_min"] = X.min(axis=1)

        return X_engineered
    
feature_pipeline = RowFeatureEngineer()

X_engineered = feature_pipeline.fit_transform(X)
X_test_engineered = feature_pipeline.transform(X_test)

print("Original shape:", X.shape)
print("Engineered shape:", X_engineered.shape)

Original shape: (4459, 4735)
Engineered shape: (4459, 4741)


# 4. Random Forest With Engineered Features

We now retrain the Random Forest model using the engineered row-level features.

The goal is to determine whether structural row summaries improve predictive performance compared to the original baseline model.

In [8]:

forest_pipeline = Pipeline([
    ("feature_engineering", RowFeatureEngineer()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

forest_scores = cross_validate(
    forest_pipeline,
    X,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    ),
    return_train_score=True
)

engineered_predictions = cross_val_predict(
    forest_pipeline,
    X,
    y,
    cv=kf
)

forest_mae = -forest_scores["test_neg_mean_absolute_error"]
forest_rmse = -forest_scores["test_neg_root_mean_squared_error"]
forest_r2 = forest_scores["test_r2"]
engineered_rmsle = rmsle(
    y,
    engineered_predictions
)

print("Random Forest With Engineered Features")
print("MAE:", forest_mae.mean())
print("RMSE:", forest_rmse.mean())
print("R²:", forest_r2.mean())
print("RMSLE:", engineered_rmsle)

Random Forest With Engineered Features
MAE: 4665446.938568786
RMSE: 6902573.465768479
R²: 0.2920077873292265
RMSLE: 1.7744333359207598


# Feature Engineering Results

Adding row-level engineered features improved Random Forest performance across all evaluation metrics.

Compared to the baseline model:

- MAE decreased
- RMSE decreased
- R² increased from approximately 0.21 to 0.29

This suggests that aggregate row statistics contain useful predictive information beyond the original anonymous feature columns.

The improvement also supports findings from the EDA notebooks:

- sparsity structure matters
- row activity patterns are informative
- global row behavior may contain predictive signal

These results show that even simple engineered features can substantially improve tree-based model performance.

# 5. Hyperparameter Tuning

We now investigate whether adjusting Random Forest hyperparameters can further improve predictive performance.

The baseline model used relatively simple default settings.

However, tree-based models are highly sensitive to parameters such as:

- tree depth
- number of trees
- minimum samples per split
- minimum samples per leaf
- feature subsampling

Careful tuning may improve generalization and reduce overfitting.

In [9]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20],
    "model__min_samples_leaf": [1, 5],
}

grid_search = GridSearchCV(
    estimator=forest_pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    refit="R2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV R²:")
print(grid_search.best_score_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Parameters:
{'model__max_depth': 20, 'model__min_samples_leaf': 5, 'model__n_estimators': 200}

Best CV R²:
0.3220338201293507


# Tuned Model Evaluation

After identifying the best Random Forest configuration through GridSearchCV, we evaluate the tuned pipeline using multiple regression metrics.

In addition to MAE, RMSE, and R², we also compute RMSLE because the target distribution is strongly right-skewed.

This helps evaluate how well the tuned model handles relative prediction errors on large target values.

In [10]:
best_forest_pipeline = grid_search.best_estimator_

tuned_scores = cross_validate(
    best_forest_pipeline,
    X,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    )
)

tuned_predictions = cross_val_predict(
    best_forest_pipeline,
    X,
    y,
    cv=kf
)

tuned_mae = -tuned_scores["test_neg_mean_absolute_error"]
tuned_rmse = -tuned_scores["test_neg_root_mean_squared_error"]
tuned_r2 = tuned_scores["test_r2"]
tuned_rmsle = rmsle(y, tuned_predictions)

print("Tuned Random Forest")
print("MAE:", tuned_mae.mean())
print("RMSE:", tuned_rmse.mean())
print("R²:", tuned_r2.mean())
print("Tuned RMSLE:", tuned_rmsle)


Tuned Random Forest
MAE: 4454989.99390581
RMSE: 6752501.562554494
R²: 0.3220338201293507
Tuned RMSLE: 1.6736048989995054


In [11]:
results_df = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "RF + Engineered Features",
        "Tuned RF + Engineered Features"
    ],

    "MAE": [
        baseline_mae.mean(),
        forest_mae.mean(),
        tuned_mae.mean()
    ],

    "RMSE": [
        baseline_rmse.mean(),
        forest_rmse.mean(),
        tuned_rmse.mean()
    ],

    "R²": [
        baseline_r2.mean(),
        forest_r2.mean(),
        tuned_r2.mean()
    ],

    "RMSLE": [
        baseline_rmsle,
        engineered_rmsle,
        tuned_rmsle
    ]
})

results_df

,Model,MAE,RMSE,R²,RMSLE
0,Baseline Random Forest,5.031415e+06,7.262026e+06,0.216445,1.911603
1,RF + Engineered Features,4.665447e+06,6.902573e+06,0.292008,1.774433
2,Tuned RF + Engineered Features,4.454990e+06,6.752502e+06,0.322034,1.673605
